# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ankitpaul6201/Fly-rank-intern-01/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Lane Selected:** `Lane 2: Refresh / Content Opportunity Scoring`

- **ML Task Type:** **Ranking & Probabilistic Opportunity Scoring** (Learning to Rank / Priority Queue Generation).
- **Why this task type?** 
  - An editorial team operates under strict weekly capacity constraints (e.g., reviewing 20–50 candidate URLs per sprint out of a total library of tens of thousands of pages). 
  - A binary classification model (`decaying: yes/no`) would flag over 13,000 pages, leaving editors with an unmanageable binary pile and no guidance on which page to update *first*.
  - Clustering groups pages by features, but lacks an actionable ordering mechanism tied to traffic recovery return on effort.
  - Therefore, we frame this as a **Ranking & Scoring** problem: predicting a continuous decay risk and traffic recovery opportunity score $S_i \in [0, 100]$ for each content item $i$, ordering the inventory so the top $K$ items represent the highest-yield editorial actions.
- **Action Supported:** Populating an automated, weekly prioritized content refresh queue for content managers and SEO editors to audit, rewrite, expand, or update.
- **Cost of a Wrong Call:**
  - *False Positive (ranking a healthy page in top 20):* Wasted editorial labor auditing pages that do not need updates (high opportunity cost).
  - *False Negative (ranking a decaying page low):* Silent search traffic loss compounding over months until search position drops below Page 2, requiring expensive ground-up content creation to recover.

In [1]:
# Section 1: Task Type & Lane Framing Summary
task_framing = {
    "Lane": "Lane 2: Refresh / Content Opportunity Scoring",
    "ML Task Type": "Ranking / Probabilistic Opportunity Scoring (Priority Queue)",
    "Primary Action": "Weekly editorial queue prioritization (Top-K refresh sprint)",
    "Unit of Analysis": "Single pseudonymized content page (content_id)",
    "Cost of False Positive": "Wasted editor hours on healthy pages (high opportunity cost)",
    "Cost of False Negative": "Compounding search decay leading to permanent traffic loss"
}

for key, val in task_framing.items():
    print(f"{key:24s}: {val}")

Lane                    : Lane 2: Refresh / Content Opportunity Scoring
ML Task Type            : Ranking / Probabilistic Opportunity Scoring (Priority Queue)
Primary Action          : Weekly editorial queue prioritization (Top-K refresh sprint)
Unit of Analysis        : Single pseudonymized content page (content_id)
Cost of False Positive  : Wasted editor hours on healthy pages (high opportunity cost)
Cost of False Negative  : Compounding search decay leading to permanent traffic loss


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

- **Target / Proxy Definition:** 
  - The target variable is the **observed relative traffic decline / impression loss** over a subsequent observation period, expressed as:
    $$\Delta \text{Imp}_{\text{rel}} = \frac{\text{impressions}_{\text{last 30d}} - \text{impressions}_{\text{prev 30d}}}{\text{impressions}_{\text{prev 30d}} + 1} \times 100$$
  - We derive a binary observed outcome flag `target_decay_flag` ($\Delta \text{Imp}_{\text{rel}} < -15.0\%$) and a continuous decay magnitude proxy `target_decay_severity` ($\max(0, -\Delta \text{Imp}_{\text{rel}})$).
- **Observed Outcome vs. Defined Rule:** 
  - **This label comes strictly from OBSERVED search outcomes measured in subsequent search console telemetry.**
  - It is **NOT** a hand-defined rule or internal product tag (such as `is_declining_label` or `health_score`). Product tags represent human assumptions and hand-selected thresholds. Predicting a product tag results in a circular model that simply memorizes an existing rule rather than learning true search behavior.
- **Feature Leakage Prevention Rule:**
  - `trend_direction` and `trend_pct` are direct mathematical transformations of the target outcome window. They are strictly **excluded from input features** and reserved exclusively for evaluation / ground truth labeling.

In [2]:
import pandas as pd
import numpy as np
import os

# Load starter dataset with fallback path support
data_path = "data/raw/content_refresh_anonymized.csv"
if not os.path.exists(data_path):
    data_path = "../../data/raw/content_refresh_anonymized.csv"

df_raw = pd.read_csv(data_path)

# Calculate observed outcome proxy from windowed search metrics
df_raw['obs_imp_change_pct'] = (df_raw['impressions_last_30d'] - df_raw['impressions_prev_30d']) / (df_raw['impressions_prev_30d'] + 1) * 100
df_raw['target_decay_flag'] = (df_raw['obs_imp_change_pct'] < -15.0).astype(int)
df_raw['target_decay_severity'] = np.maximum(0, -df_raw['obs_imp_change_pct'])

print("=== TARGET & PROXY SUMMARY ===")
print(f"Total Rows Analyzed: {len(df_raw):,}")
print(f"Observed Decay Flag Rate (Imp Drop > 15%): {df_raw['target_decay_flag'].mean():.2%}")
print(f"Mean Observed Imp Change (%): {df_raw['obs_imp_change_pct'].mean():.2f}%")
print(f"Median Observed Imp Change (%): {df_raw['obs_imp_change_pct'].median():.2f}%")
print("\nTarget Label Distribution:")
print(df_raw['target_decay_flag'].value_counts(normalize=True).rename({1: "Decaying (1)", 0: "Stable/Growing (0)"}))

=== TARGET & PROXY SUMMARY ===
Total Rows Analyzed: 30,000
Observed Decay Flag Rate (Imp Drop > 15%): 57.75%
Mean Observed Imp Change (%): 1188.36%
Median Observed Imp Change (%): -25.61%

Target Label Distribution:
target_decay_flag
Decaying (1)          0.577467
Stable/Growing (0)    0.422533
Name: proportion, dtype: float64


## 3. Success metric

*One metric you can defend. What number means 'good'?*

- **Primary Success Metric:** **Precision@K (specifically Precision@50 and Precision@20)**.
  - $$\text{Precision}@K = \frac{\text{Number of truly decaying & recoverable pages in top } K \text{ ranked recommendations}}{K}$$
- **Why Precision@K is the single most defensible metric:**
  - Content teams have fixed weekly bandwidth (e.g. reviewing $K=50$ pages per week). 
  - Standard accuracy or ROC-AUC evaluates performance across the entire 30,000-page dataset, which is irrelevant to an editor who only ever looks at the top of the queue.
  - Precision@50 directly measures operational efficiency: a Precision@50 of **85%** guarantees that 42 out of 50 editor reviews target genuine content decay opportunities.
- **Secondary Metrics:**
  - **NDCG@50 (Normalized Discounted Cumulative Gain):** Measures whether the most severe decay cases are ranked at the very top of the 50-item list.
  - **Baseline to Beat:** A naive rule baseline (e.g., sorting by `days_since_last_update` or `content_age_days`), which achieves ~66%–84% Precision@50 on this dataset.

In [3]:
# Code to compute Precision@K and NDCG@K for ranking evaluation

def compute_precision_at_k(y_true, y_score, k=50):
    """Calculates Precision@K for a given score ranking."""
    top_k_idx = np.argsort(y_score)[::-1][:k]
    return np.mean(y_true.iloc[top_k_idx])

def compute_ndcg_at_k(y_true, y_score, k=50):
    """Calculates NDCG@K for ranking quality."""
    top_k_idx = np.argsort(y_score)[::-1][:k]
    gains = y_true.iloc[top_k_idx].values
    discounts = np.log2(np.arange(2, k + 2))
    dcg = np.sum(gains / discounts)
    
    ideal_gains = np.sort(y_true.values)[::-1][:k]
    idcg = np.sum(ideal_gains / discounts)
    return dcg / idcg if idcg > 0 else 0.0

# Filter to Lane 2 active slice
active_slice = df_raw[df_raw['impressions_90d'] >= 100].copy()

# Baseline 1: Random ranking
np.random.seed(42)
p50_random = compute_precision_at_k(active_slice['target_decay_flag'], np.random.rand(len(active_slice)), k=50)

# Baseline 2: Simple heuristic rule (Sort by content age - oldest first)
p50_age = compute_precision_at_k(active_slice['target_decay_flag'], active_slice['content_age_days'], k=50)

# Baseline 3: Simple heuristic rule (Days since last update)
p50_fresh = compute_precision_at_k(active_slice['target_decay_flag'], active_slice['days_since_last_update'], k=50)

print("=== SUCCESS METRIC BENCHMARKS (Precision@50) ===")
print(f"Random Ranking Baseline       : {p50_random:.2%}")
print(f"Rule Baseline (Content Age)   : {p50_age:.2%}")
print(f"Rule Baseline (Days Unupdated): {p50_fresh:.2%}")
print(f"Target ML Benchmark Goal      : > 90.00% Precision@50")

=== SUCCESS METRIC BENCHMARKS (Precision@50) ===
Random Ranking Baseline       : 72.00%
Rule Baseline (Content Age)   : 84.00%
Rule Baseline (Days Unupdated): 66.00%
Target ML Benchmark Goal      : > 90.00% Precision@50


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

- **Unit of Analysis (Grain):** **One row = One pseudonymized content item (`content_id`)** belonging to a specific client site (`client_id`).
- **Lane Slice Definition:** We slice the starter dataset to include content pages with active search demand (`impressions_90d >= 100`). This filters out 7,994 unindexed or zero-traffic stub pages, leaving **22,006 active content pages** across 30 pseudonymized client sites.
- **Data Integrity Verification:** Each row is uniquely identified by `content_id` (`nunique() == len(lane_slice)`).

In [4]:
# Load, slice, and inspect Lane 2 Unit of Analysis DataFrame

df_full = pd.read_csv(data_path)

# Filter slice for Lane 2: Pages with visible search demand (impressions_90d >= 100)
lane_slice = df_full[df_full['impressions_90d'] >= 100].copy().reset_index(drop=True)

# Target computation
lane_slice['obs_imp_change_pct'] = (lane_slice['impressions_last_30d'] - lane_slice['impressions_prev_30d']) / (lane_slice['impressions_prev_30d'] + 1) * 100
lane_slice['target_decay_risk'] = np.maximum(0, -lane_slice['obs_imp_change_pct'])

# Grain check
assert lane_slice['content_id'].nunique() == len(lane_slice), "Grain violation: content_id is not unique!"

print(f"=== LANE 2 SLICE SUMMARY ===")
print(f"Total Inventory Rows       : {len(df_full):,}")
print(f"Lane 2 Active Slice Rows   : {len(lane_slice):,} (Grain: 1 row = 1 content_id)")
print(f"Pseudonymized Client Count : {lane_slice['client_id'].nunique()}")

# Display the real slice dataframe with key features & target column
preview_cols = [
    'content_id', 'client_id', 'content_type', 
    'impressions_90d', 'clicks_90d', 'ctr', 'avg_position', 
    'content_age_days', 'days_since_last_update', 'target_decay_risk'
]

print("\nUnit of Analysis DataFrame Preview (First 5 rows):")
display(lane_slice[preview_cols].head())

=== LANE 2 SLICE SUMMARY ===
Total Inventory Rows       : 30,000
Lane 2 Active Slice Rows   : 22,006 (Grain: 1 row = 1 content_id)
Pseudonymized Client Count : 30

Unit of Analysis DataFrame Preview (First 5 rows):


,content_id,client_id,content_type,impressions_90d,clicks_90d,ctr,avg_position,content_age_days,days_since_last_update,target_decay_risk
0,content_304f48230142,client_f369cb89fc,keyword article,3803,29,0.76,10.6,187,20,41.396761
1,content_a1fb4e703a9e,client_4e07408562,keyword article,15320,7,0.05,20.3,445,25,57.707911
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,12581,11,0.09,36.5,141,20,60.870279
3,content_331d6c4de07b,client_19581e27de,keyword article,11751,58,0.49,6.2,463,22,13.786546
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,19140,24,0.13,44.0,263,14,34.728033


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule (e.g. `IF days_since_last_update > 180 AND ctr < 1.0% THEN flag_refresh`) fails in complex search environments for four structural reasons:

1. **Position-Dependent CTR Non-Linearity:** 
   - A CTR of 0.8% for a page at `avg_position = 1.5` is an alarming underperformance (CTR deficit). 
   - The same CTR of 0.8% for a page at `avg_position = 18.2` is actually an overperformance. A static rule with `ctr < 1.0%` treats both identically, creating false alarms on page-2 rankings while missing decaying top-3 content.
2. **Multi-Signal Interaction Effects:** 
   - Content decay is not caused by age alone. It emerges from subtle interactions between search volume, position tier, CTR deficit, engagement rates, scroll depth, and AI traffic displacement (`ai_traffic_pct`). Hand-coded if-statements cannot learn the weights or interaction thresholds across 15+ continuous variables.
3. **Client & Content Type Heterogeneity:** 
   - Baseline engagement and update frequencies vary widely between `keyword article` and `landing page` content types. A fixed rule imposes arbitrary global thresholds that fail on niche content.
4. **Binary Decision vs. Continuous Value Score:** 
   - A rule outputs a blunt YES/NO flag without ranking severity. ML computes a continuous expected decay score $S_i$, enabling precise top-$K$ prioritization aligned with available editorial capacity.

In [5]:
# Empirical demonstration: Static Rule vs. Multi-Signal Heuristics

# 1. Evaluate a typical static hand-written rule: (CTR < 0.5% & days_since_last_update > 90)
lane_slice['static_rule_flag'] = (lane_slice['ctr'] < 0.5) & (lane_slice['days_since_last_update'] > 90)

# Compute target decay flag (Imp drop > 15%)
lane_slice['target_decay_flag'] = (lane_slice['obs_imp_change_pct'] < -15.0).astype(int)

# Precision@50 of static rule
rule_p50 = compute_precision_at_k(lane_slice['target_decay_flag'], lane_slice['static_rule_flag'].astype(float), k=50)

# 2. Multi-Signal Composite Score (Modeling position-adjusted CTR deficit + Content Age)
expected_ctr = np.maximum(0.1, 10.0 / (lane_slice['avg_position'].replace(0, 50) + 1))
ctr_deficit = np.maximum(0, expected_ctr - lane_slice['ctr'])
lane_slice['multi_signal_score'] = ctr_deficit * np.log1p(lane_slice['content_age_days'])

multi_p50 = compute_precision_at_k(lane_slice['target_decay_flag'], lane_slice['multi_signal_score'], k=50)

print("=== EMPIRICAL COMPARISON: FIXED RULE vs MULTI-SIGNAL SCORING ===")
print(f"Static Rule Trigger Count      : {lane_slice['static_rule_flag'].sum():,} pages")
print(f"Static Rule Precision@50       : {rule_p50:.2%}")
print(f"Multi-Signal Score Precision@50: {multi_p50:.2%}")
print(f"Performance Gain over Rule    : +{(multi_p50 - rule_p50)*100:.2f} percentage points")
print("\nConclusion: Multi-signal scoring captures position-CTR interactions that static IF-THEN rules completely miss.")

=== EMPIRICAL COMPARISON: FIXED RULE vs MULTI-SIGNAL SCORING ===
Static Rule Trigger Count      : 7,005 pages
Static Rule Precision@50       : 78.00%
Multi-Signal Score Precision@50: 88.00%
Performance Gain over Rule    : +10.00 percentage points

Conclusion: Multi-signal scoring captures position-CTR interactions that static IF-THEN rules completely miss.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.